In [1]:
import torch
import os
from pathlib import Path
import numpy as np
import time

from ase.optimize import LBFGS
from ase.filters import FrechetCellFilter
from ase.md import VelocityVerlet, MDLogger
from ase.md.nptberendsen import NPTBerendsen
from ase.md.langevin import Langevin
from ase.io import Trajectory, read
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary, ZeroRotation
from ase.units import fs, bar, kB

import openmm.app as app

from cmm.interfaces import CMMCalculator
from cmm.ffxml import ForceFieldXML
from cmm.topology import Topology
from cmm.units import BOHR2NM, BOHR2ANG, HARTREE2KCAL

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64)

In [3]:
ff_path = os.path.join(Path.home(), "dev/python_development/pyCMM/scripts/ion_water_refit.xml")
h2o_f_pdb_path = os.path.join(Path.home(), 'dev/python_development/pyCMM/notebooks/data/h2o_f_dimer.pdb')
h2o_cl_pdb_path = os.path.join(Path.home(), 'dev/python_development/pyCMM/notebooks/data/h2o_cl_dimer.pdb')

ff = ForceFieldXML(ff_path, device=device)
pdb = app.PDBFile(h2o_f_pdb_path)
top = Topology.fromOpenmm(pdb.topology, device)

system = ff.parametrize(top, use_fd_morse=True, cutoff_sr=6.0, use_lr_dispersion=False, use_hardness_change=False, use_polarization=True, use_ewald=False)
coords = torch.tensor(pdb.getPositions(asNumpy=True)._value / BOHR2NM, device=device, requires_grad=True)
box = torch.tensor([[vec.x / BOHR2NM, vec.y / BOHR2NM, vec.z / BOHR2NM] for vec in pdb.topology.getPeriodicBoxVectors()], device=device, requires_grad=True)

energies = system.getEnergy(coords, box)

/home/heindelj/miniforge3/envs/pycmm/lib/python3.12/site-packages/torch/nested/__init__.py:226: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  return _nested.nested_tensor(


In [4]:
for key in energies.keys():
    print(f"{key}: {energies[key].item() * HARTREE2KCAL:.4f}")

bond: 4.5685
angle: 0.0571
torsion: 0.0000
bond_bond: 0.0022
bond_angle: -0.1173
angle_angle: 0.0000
torsion_bond: 0.0000
torsion_angle: 0.0000
torsion_angle_angle: 0.0000
perm_elec: 0.0000
pol: -0.0000
ct_direct: 0.0000
xpol: -3.0165
pauli: 44.8696
disp: -4.2862
total: 42.0774
